# Anvil on a GPU: the M6 generation eval

This notebook runs the parts of the evals that need a GPU:

- **M6 generation eval** (`eval.generate wikisql`): a LoRA fine-tune of
  Qwen2.5-0.5B-Instruct on 1,000 WikiSQL teacher labels, scored by execution match against
  the teacher (parity), then merged and exported.
- **ModernBERT frontier** (optional, `eval.frontier`): the M3 frontier with the full-size
  student.

**Runtime:** Runtime -> Change runtime type -> **T4 GPU**. My estimate is under an hour for
the generation eval and several hours for the frontier; the run records its real wall time.

**Cost:** with no `ANTHROPIC_API_KEY` the teacher is the dataset's gold labels (model id
`gold`) and the run spends $0. To label with `claude-haiku-5-5` instead, add the key as a
Colab secret named `ANTHROPIC_API_KEY` (the key icon in the left bar) and allow this
notebook to read it. Labels already committed under `eval/labels/` are replayed for free.

In [ ]:
REPO = "https://github.com/TheAsianFish/anvil"
REF = "main"

!nvidia-smi
# The repo is private until it is published: either put a token in the URL
# (https://<token>@github.com/...) or upload a zip of the repo and unzip it to /content/anvil.
!git clone --branch {REF} {REPO} /content/anvil
%cd /content/anvil
!pip install -q uv
# The lock file pins CPU torch. Install everything else from the lock, then the CUDA build of
# the same torch version (2.14.1, as in uv.lock).
!uv sync --all-extras --group eval --no-install-package torch
!uv pip install torch==2.14.1 --index-url https://download.pytorch.org/whl/cu126

import os

# Without this every `uv run` would re-sync the environment and put the CPU torch back.
os.environ["UV_NO_SYNC"] = "1"
!uv run python -c "import torch; assert torch.cuda.is_available(), 'no GPU'; print(torch.__version__, torch.cuda.get_device_name(0))"

In [ ]:
# Optional: the Anthropic key from Colab secrets. Without it the gold labels are the teacher.
try:
    from google.colab import userdata

    key = userdata.get("ANTHROPIC_API_KEY")
except Exception:  # not on Colab, no such secret, or access not granted
    key = None
if key:
    os.environ["ANTHROPIC_API_KEY"] = key
print("teacher:", "claude-haiku-5-5" if key else "gold (no key, $0)")

In [ ]:
!uv run --group eval python -m eval.generate wikisql --teacher auto
!uv run python -m eval.render

## Optional: the ModernBERT frontier

This one is long (several hours on a T4). Skip it unless you want the full-size frontier.

In [ ]:
!uv run --group eval python -m eval.frontier --teacher auto
!uv run python -m eval.render

## Download the raw output

Every number in RESULTS.md has to come from raw output committed under `eval/results/`.
Download the zip, unzip it at the root of the repo, run `make results` and commit the
directories together with the updated RESULTS.md.

In [ ]:
import glob
import shutil
import tempfile

dirs = sorted(glob.glob("eval/results/generate-*") + glob.glob("eval/results/frontier-*"))
print("\n".join(dirs))
staging = tempfile.mkdtemp()
for d in dirs:
    shutil.copytree(d, os.path.join(staging, d))
archive = shutil.make_archive("/content/anvil-results", "zip", staging)

from google.colab import files

files.download(archive)